# 第 7 讲实验：Collectives 与分布式训练基础

本 Notebook 用 CPU 模拟 collective 的数学语义，并用 alpha-beta 模型估算 ring all-reduce 通信量。它不会启动真实多进程，也不会发送网络消息；不能替代 torch.distributed / NCCL 测试。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_07").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_07"))
from collectives_lab import (
    all_reduce_sum, all_reduce_mean, ring_all_reduce_sum,
    weighted_mean_gradient, ring_all_reduce_bytes_per_rank,
    ring_all_reduce_time_seconds,
)
print("PyTorch:", torch.__version__, "| simulation device: CPU")

## 实验 1：All-reduce 与 ring 模拟

每个 rank 起初持有自己的 tensor。All-reduce sum 之后，所有 rank 都应获得相同的逐元素总和。Ring 模拟将 flat tensor 分成 P 个等长 chunk，先 reduce-scatter，再 all-gather。

In [ ]:
rank_values = [
    torch.tensor([1.0, 2.0, 3.0, 4.0]),
    torch.tensor([10.0, 20.0, 30.0, 40.0]),
    torch.tensor([100.0, 200.0, 300.0, 400.0]),
    torch.tensor([1000.0, 2000.0, 3000.0, 4000.0]),
]
reference = all_reduce_sum(rank_values)
ring = ring_all_reduce_sum(rank_values)
print("reference sum:", reference[0].tolist())
print("ring results:", [x.tolist() for x in ring])
assert all(torch.equal(x, reference[0]) for x in ring)
assert all(torch.equal(x, ring[0]) for x in ring)

## 实验 2：不等 local batch 时的梯度权重

若 g_r 是 rank r 的样本平均梯度，且各 rank 的样本数不同，正确的全局样本平均是按样本数加权，而不是直接对 rank 平均。语言模型场景还要考虑有效 token 数和 loss mask。

In [ ]:
g0 = torch.tensor([1.0, 3.0])  # rank 0: 1 sample
g1 = torch.tensor([5.0, 7.0])  # rank 1: 3 samples
rank_mean = all_reduce_mean([g0, g1])[0]
sample_mean = weighted_mean_gradient([g0, g1], [1, 3])
print("naive rank mean:", rank_mean.tolist())
print("sample-weighted mean:", sample_mean.tolist())
assert torch.equal(rank_mean, torch.tensor([3.0, 5.0]))
assert torch.equal(sample_mean, torch.tensor([4.0, 6.0]))
plt.figure(figsize=(6, 3.5))
labels = ["rank mean", "sample-weighted"]
plt.bar(labels, [rank_mean[0].item(), sample_mean[0].item()])
plt.ylabel("First gradient component")
plt.title("Unequal local batch: rank mean is not sample mean")
plt.tight_layout()
plt.show()

## 实验 3：Ring all-reduce 通信量与简化时间模型

每 rank 发送量近似为 2(P-1)/P × tensor_bytes。alpha-beta 时间模型近似包含每个阶段的启动延迟与带宽项，但不包含拥塞、拓扑、通信/计算重叠和库的算法选择。以下参数均为教学合成值。

In [ ]:
tensor_bytes = 1024 * 1024  # 1 MiB
world_sizes = [1, 2, 4, 8, 16]
per_rank_mib = [
    ring_all_reduce_bytes_per_rank(tensor_bytes, 1, p) / 1024**2
    for p in world_sizes
]
alpha = 5e-6
bandwidth = 20e9  # synthetic 20 GB/s, not a measured link
times_us = [
    ring_all_reduce_time_seconds(alpha, bandwidth, tensor_bytes, p) * 1e6
    for p in world_sizes
]
for p, volume, time_us in zip(world_sizes, per_rank_mib, times_us):
    print(f"P={p:2d}: sent/rank={volume:.3f} MiB, alpha-beta estimate={time_us:.2f} us")
plt.figure(figsize=(6, 3.5))
plt.plot(world_sizes, per_rank_mib, marker="o")
plt.xlabel("World size P")
plt.ylabel("Bytes sent per rank (MiB)")
plt.title("Ring all-reduce per-rank communication volume")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("These are formulas with synthetic inputs, not measured network performance.")

## 复盘

1. Reduce-scatter 与 all-gather 分别做什么？
2. 为什么 ring all-reduce 的发送量公式含有 2(P-1)/P？
3. 为什么对 rank 做简单平均在 local batch 不等时会有偏差？
4. 为什么 CPU 模拟通过不代表真实 Gloo/NCCL 通信可靠或更快？
5. 真实分布式测试需要记录哪些环境与拓扑信息？